In [ ]:
import pandas as pd
import numpy as np
import os
import re
from openpyxl import load_workbook


In [ ]:
ruta =  "Febrero_2021/26-2-_21.xlsx"

In [ ]:

color_a_persona = {
    "FF002060": "D",
    "FF1BD9ED": "L",
    "FF21FF85": "T",
    "FF660066": "DO",
    "FF996600": "K",
    "FFFF6600": "A",
}

wb = load_workbook(ruta)
ws = wb.active

# Buscar columna REP
columna_rep = None
fila_encabezado = None

for fila in ws.iter_rows():
    for celda in fila:
        if celda.value and str(celda.value).strip().upper() == "REP":
            columna_rep = celda.column_letter
            fila_encabezado = celda.row
            break
    if columna_rep:
        break

if columna_rep is None:
    raise ValueError("No se encontró la columna REP")

# Procesar filas debajo del encabezado
for fila in range(fila_encabezado + 1, ws.max_row + 1):
    celda = ws[f"{columna_rep}{fila}"]
    color = celda.fill.start_color.rgb

    if color and color != "00000000" and color in color_a_persona:
        celda.value = color_a_persona[color]

wb.save(ruta)
#wb.save("/content/drive/MyDrive/Proyecto_Diego/cleaned_enero_2021/prueba.xlsx")

In [ ]:
df=pd.read_excel(ruta, skiprows=7)


In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
df.head(91)

In [ ]:
nombre_archivo = os.path.basename(ruta)            # '03-01-2026.csv'
f_str = os.path.splitext(nombre_archivo)[0]    # '03-01-2026'

# eliminar cualquier cosa que NO sea número o guión
fecha_str = re.sub(r"[^0-9\-]", "", f_str)
fecha = pd.to_datetime(fecha_str, format="%d-%m-%y").strftime("%d-%m-%Y")


In [ ]:
df.shape

In [ ]:
if "Total" in df.columns:
    df.drop(columns=df.loc[:, "Total":].columns, inplace=True)


In [ ]:
df.rename(columns={"M/Tortilla":"Masa_tortilla","M/Tamales":"Masa_tamal"},inplace=True)

In [ ]:
df["Repartidor"]=""

In [ ]:
df["REP"].value_counts()

In [ ]:
df["REP"]=df["REP"].astype(str)

In [ ]:
for i in df.index:
    if df.loc[i,"REP"]=="D":
        df.loc[i,"Repartidor"]="Diego"
    elif df.loc[i,"REP"]=="L":
        df.loc[i,"Repartidor"]="Laura"
    elif df.loc[i,"REP"]=="A":
        df.loc[i,"Repartidor"]="Alex"
    elif df.loc[i,"REP"]=="T":
        df.loc[i,"Repartidor"]="Toker"
    elif df.loc[i,"REP"]=="K":
        df.loc[i,"Repartidor"]="Kevin"
    elif df.loc[i,"REP"]=="DO":
        df.loc[i,"Repartidor"]="Donaldo"
    else:
        df.loc[i,"Repartidor"]="NA"

In [ ]:
df["Repartidor"].value_counts()

In [ ]:
df.drop(columns={"REP"},inplace=True)

In [ ]:
df["Fecha"] = fecha

In [ ]:
df["Entrega"]=df["Entrega"].fillna(0)
df["Devolución"]=df["Devolución"].fillna(0)
df["Masa_tortilla"]=df["Masa_tortilla"].fillna(0)
df["Masa_tamal"]=df["Masa_tamal"].fillna(0)
df["Sin epecificar"]=df["Sin epecificar"].fillna(0)
df["T. Chica"]=df["T. Chica"].fillna(0)

In [ ]:
df.dropna(axis=0, how="any",subset="Cliente",inplace=True)

In [ ]:
df.head(100)

In [ ]:
df["Entrega"]=df["Entrega"]+df["T. Chica"]
df.drop(columns={"T. Chica"},inplace=True)


#df["Entrega"]=df["Entrega"]+df["T. Especial"]
#df.drop(columns={"T. Especial"},inplace=True)

In [ ]:
df.reset_index(drop=True, inplace=True)

In [ ]:
df.head(100)

In [ ]:
#idx_corte = 42
idx_corte = df.index[df["Cliente"] == "SEGUNDA VUELTA"][0]

df_1 = df.iloc[:idx_corte].copy()
df_2 = df.iloc[idx_corte:].copy()


In [ ]:
df_1.drop(columns={"Sin epecificar"},inplace=True)
#df_2.drop(columns={"Sin epecificar"},inplace=True)


#df_1["Entrega"]=df_1["Entrega"]+df_1["T.Chica"]
#df_1.drop(columns={"T.Chica"},inplace=True)



In [ ]:
#df_2["Entrega"]=df_2["Entrega"]+df_2["T.Chica"]
#df_2.drop(columns={"T.Chica"},inplace=True)

In [ ]:
df_2.drop(columns={"Devolución"},inplace=True)

In [ ]:
df_2.rename(columns={"Masa_tortilla":"Devolución","Masa_tamal":"Masa_tortilla","Sin epecificar":"Masa_tamal"},inplace=True)

In [ ]:
df_f = pd.DataFrame(np.vstack([df_1.to_numpy(), df_2.to_numpy()]), columns=df_1.columns)


In [ ]:
df_f.drop(df.index[-1], inplace= True)

In [ ]:
df = df_f[~df_f["Cliente"].isin(["SEGUNDA VUELTA", "Cliente", "Aquí"])]

In [ ]:
df["Cliente"]=df["Cliente"].astype(str)
df["Entrega"]=df["Entrega"].astype(float)
df["Devolución"]=df["Devolución"].astype(float)
df["Masa_tortilla"]=df["Masa_tortilla"].astype(float)
df["Masa_tamal"]=df["Masa_tamal"].astype(float)
df["Repartidor"]=df["Repartidor"].astype(str)

In [ ]:
df.reset_index(drop=True, inplace=True)

In [ ]:
ruta_salida = f"cleaned_febrero_2021/{fecha}.csv"
df.to_csv(ruta_salida, index=False)
